When invoking a graph with a checkpointer, you must specify a thread_id as part of the configurable portion of the config

In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from typing import Annotated
from typing_extensions import TypedDict
from operator import add

class State(TypedDict):
    foo: str
    bar: Annotated[list[str], add]

def node_a(state: State):
    return {"foo": "a", "bar": ["a"]}

def node_b(state: State):
    return {"foo": "b", "bar": ["b"]}


workflow = StateGraph(State)
workflow.add_node(node_a)
workflow.add_node(node_b)
workflow.add_edge(START, "node_a")
workflow.add_edge("node_a", "node_b")
workflow.add_edge("node_b", END)

checkpointer = InMemorySaver()
graph = workflow.compile(checkpointer=checkpointer)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}
state = graph.invoke({"foo": "", "bar":[]}, config)

In [2]:
state

{'foo': 'b', 'bar': ['a', 'b']}

You can access the checkpoint namespace from within a node via the config:

In [3]:
def node_c(state: State, config: RunnableConfig):
    checkpoint_ns = config["configurable"]["checkpoint_ns"] # type: ignore
    return {"foo": checkpoint_ns}

workflow2 = StateGraph(State)
workflow2.add_node(node_a)
workflow2.add_node(node_c)
workflow2.add_edge(START, "node_a")
workflow2.add_edge("node_a", "node_c")
workflow2.add_edge("node_c", END)
graph2 = workflow2.compile(checkpointer=InMemorySaver())

state2 = graph2.invoke({"foo": "", "bar":[]}, {"configurable": {"thread_id": "2"}})

In [4]:
state2

{'foo': 'node_c:9bae89ef-88ec-0da6-4d6b-2e0a4fcd6387', 'bar': ['a']}